# HOMER: functional estimates and perturbation stability

Functional observations are represented by 12 Fourier coefficients, with concentrated or diffuse covariance spectra. One experiment studies fixed-threshold inference for three contrasts. A separate paired experiment compares point-estimation error under clean data and perturbations concentrated in a few blocks or dispersed across raw observations.

The distinction matters: the adaptive fits in the stability experiment do not inherit the fixed-threshold inference result.


## Choose the results to read

`paper` reads the compact manuscript summaries without rerunning experiments. Set `MODE` to `smoke` for a short numerical run or `full` for manuscript replication counts. Fresh outputs go to `runs/`; the saved figures below are displayed only in `paper` mode.


In [ ]:
MODE = "paper"  # "paper", "smoke", or "full"
STUDIES = ['functional']


In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "results").is_dir() and (p / "notebooks").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("Open this notebook from the repository root or notebooks directory.")
if MODE not in {"paper", "smoke", "full"}:
    raise ValueError("MODE must be 'paper', 'smoke', or 'full'.")

if MODE == "paper":
    RESULTS = ROOT / "results"
    display(Markdown("**Reading the saved manuscript results.** No experiments are rerun."))
else:
    subprocess.run(
        [sys.executable, str(ROOT / "code" / "reproduce.py"), "--mode", MODE,
         "--study", *STUDIES, "--resume"],
        cwd=ROOT, check=True,
    )
    RESULTS = ROOT / "runs" / MODE / "results"
    display(Markdown(f"**Reading {MODE} results** from `runs/{MODE}/results/`."))

pd.set_option("display.max_columns", 12)
pd.set_option("display.precision", 4)


## Inference for functional contrasts

The saved experiment has 12 cells and 500 fits per cell, with three contrast records per fit. These are population-mean coverage summaries with pointwise Monte Carlo intervals.


In [ ]:
inference = pd.read_csv(RESULTS / "functional_inference_summary.csv")
display(inference[["spectrum", "distribution", "k", "m", "contrast",
                   "reps", "coverage", "mc_low", "mc_high"]])


## Paired point-estimation comparisons

Methods share clean observations, affected indices, and perturbation magnitudes. The display names below follow the manuscript; the saved CSV keys remain unchanged. Normal Monte Carlo intervals for Student-t4 MSE comparisons are omitted because a finite fourth moment is not available.


In [ ]:
point = pd.read_csv(RESULTS / "functional_point_summary.csv")
DISPLAY_METHOD = {
    "PH(1)": "HOMER(1)", "PH(16)": "HOMER(16)",
    "Adaptive(2)": "Adaptive HOMER", "Canonical(2)": "Canonical adaptive HOMER",
}
point["display_method"] = point["method"].replace(DISPLAY_METHOD)
selected = point.loc[
    point["method"].isin(["Mean", "MOM", "PH(1)", "PH(16)", "Adaptive(2)"]),
    ["spectrum", "distribution", "mechanism", "magnitude", "display_method",
     "reps", "mse", "q95", "mean_realized_block_fraction"],
]
display(selected.reset_index(drop=True))


In [ ]:
# Inference failures are recorded once per contrast, so deduplicate the fits' cells.
checks = inference.drop_duplicates(["spectrum", "distribution", "a", "k", "m"])
display(checks[["solver_failures", "covariance_failures", "certificate_failures"]]
        .sum().rename("total").to_frame())


## Manuscript figure

The first two panels summarize the functional simulations. The third is the separate wearable illustration, discussed in `covariance_wearable.ipynb`.

In [ ]:
if MODE == "paper":
    display(Image(filename=str(ROOT / "figures" / "exp_functional_wearable.png")))
else:
    display(Markdown(
        "Fresh-run summaries are shown above. To regenerate all manuscript-style "
        "figures from a complete run, use `python code/reproduce.py --mode " + MODE + " --figures`. "
        "Figure generation requires pdfLaTeX, PGF, and Poppler."
    ))


## Interpretation

The comparison examines which perturbations the aggregation step can tolerate. A raw-observation contamination fraction does not determine the fraction of affected blocks: dispersed observations can touch many blocks. The recorded affected-block fraction makes this difference visible.

Fixed-threshold mean inference and adaptive point estimation serve different purposes in the paper. Saved paired differences and bootstrap summaries support comparisons beyond the plotted curves. Full runs regenerate the replicate records and pairing arrays.
